# Deep CFR Paper Implementation Verification

This notebook systematically verifies that the implementation matches the Deep CFR paper:
- **Algorithm 1**: DEEPCFR main loop
- **Algorithm 2**: TRAVERSE function  
- **Section 5.1**: Network architecture
- **Section 5.2**: Training parameters

In [1]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.abspath('')))
sys.path.insert(0, os.path.abspath('.'))

import torch
import torch.nn.functional as F
import numpy as np
from collections import defaultdict

# Track test results
TEST_RESULTS = {}

def record_test(name, passed, details=""):
    """Record test result."""
    TEST_RESULTS[name] = {'passed': passed, 'details': details}
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {name}")
    if details:
        print(f"       {details}")
    return passed

## Test 1: Network Initialization (Algorithm 1, Line 578)

**Paper**: "Initialize each player's advantage network V(I,a|θp) with parameters θp so that it returns 0 for all inputs."

**Verification**: Fresh network should output near-zero values.

In [2]:
from network.model import DeepCFRModule
from utils.infoset_parser import parse_infoset_to_network_input

print("=" * 60)
print("TEST 1: Network Initialization")
print("=" * 60)

# Create fresh network
network = DeepCFRModule(
    nhandcards=3,
    nboardcards=5,
    n_action_history=20,
    nresponses=9,
    dim=256
)

# Test with sample inputs - use proper infoset format
test_infoset = "S0|H:14s0,13s0,10s1|B:|A:"
sample_cc, sample_ah = parse_infoset_to_network_input(test_infoset)

with torch.no_grad():
    output = network([sample_cc], [sample_ah])

max_output = torch.abs(output).max().item()
mean_output = torch.abs(output).mean().item()

print(f"\nNetwork output shape: {output.shape}")
print(f"Output values: {output[0].numpy()}")
print(f"Max absolute output: {max_output:.6f}")
print(f"Mean absolute output: {mean_output:.6f}")

# Paper requirement: outputs should be 0 (we use near-zero for gradient flow)
threshold = 0.5  # Generous threshold for near-zero
passed = max_output < threshold

record_test(
    "1. Network initialization outputs near-zero",
    passed,
    f"Max output {max_output:.4f} < {threshold} threshold"
)

TEST 1: Network Initialization


AttributeError: 'list' object has no attribute 'get_canonical_hand_tensor'

## Test 2: Network-Guided Traversal (Algorithm 2, Lines 611, 619)

**Paper**: "Compute strategy σt(I) from predicted advantages V(I(h),a|θp) using regret matching."

**Verification**: During traversal, the network must be called to predict regrets.

In [ ]:
from core.mccfr import MCCFR
from core.integration import NetworkMCCFRIntegration

print("=" * 60)
print("TEST 2: Network-Guided Traversal")
print("=" * 60)

# Create MCCFR and network integration
mccfr = MCCFR()
network_p0 = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
network_p1 = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)

integration_p0 = NetworkMCCFRIntegration(network_p0, mccfr)
integration_p1 = NetworkMCCFRIntegration(network_p1, mccfr)

network_integrations = {0: integration_p0, 1: integration_p1}

# Create initial state
state = mccfr.create_initial_state()
mccfr.set_iteration(1)

# Test that get_network_regrets is callable
active_player = state.button % 2
try:
    regrets = integration_p0.get_network_regrets(state, active_player)
    print(f"\nNetwork regrets for player {active_player}:")
    for action, value in regrets.items():
        print(f"  {action}: {value:.4f}")
    
    # Verify regrets are returned for legal actions
    legal_actions = mccfr.get_legal_actions_list(state)
    legal_keys = [mccfr.action_to_key(a, state, active_player) for a in legal_actions]
    
    has_regrets = len(regrets) > 0
    record_test(
        "2a. Network returns regrets for state",
        has_regrets,
        f"Got {len(regrets)} regret values"
    )
except Exception as e:
    record_test("2a. Network returns regrets for state", False, str(e))

In [ ]:
# Test that external_sampling_with_network actually uses the network
print("\nTesting external_sampling_with_network...")

# Check if the method exists
has_method = hasattr(mccfr, 'external_sampling_with_network')
record_test(
    "2b. external_sampling_with_network method exists",
    has_method,
    "Method found in MCCFR class" if has_method else "Method NOT found!"
)

if has_method:
    # Run a short traversal
    mccfr.clear_advantage_memory()
    mccfr.clear_strategy_memory()
    
    state = mccfr.create_initial_state()
    try:
        utility = mccfr.external_sampling_with_network(
            state, 
            traversing_player=0, 
            network_integrations=network_integrations,
            collect_deep_cfr_samples=True
        )
        
        # Check samples were collected
        p0_samples = mccfr.get_advantage_samples(0)
        strategy_samples = mccfr.get_strategy_samples()
        
        print(f"\nTraversal utility: {utility:.4f}")
        print(f"P0 advantage samples: {len(p0_samples)}")
        print(f"Strategy samples: {len(strategy_samples)}")
        
        record_test(
            "2c. Network traversal collects samples",
            len(p0_samples) > 0 or len(strategy_samples) > 0,
            f"Collected {len(p0_samples)} advantage, {len(strategy_samples)} strategy samples"
        )
    except Exception as e:
        record_test("2c. Network traversal collects samples", False, str(e))

## Test 3: Instantaneous Regrets Formula (Algorithm 2, Lines 615-616)

**Paper**: `r̃(I,a) = v(a) - Σ σ(a') · v(a')`

**Verification**: Check that regrets are computed as action_value - node_value

In [ ]:
import inspect
from core.mccfr import MCCFR

print("=" * 60)
print("TEST 3: Instantaneous Regrets Formula")
print("=" * 60)

# Get source code of external_sampling_with_network
source = inspect.getsource(MCCFR.external_sampling_with_network)

# Check for the regret formula pattern
has_action_values = "action_values[" in source
has_node_value = "node_value" in source
has_regret_subtraction = "action_values[action_key] - node_value" in source or \
                         "action_values[action_key]-node_value" in source

print("\nChecking regret computation in external_sampling_with_network:")
print(f"  - Has action_values dict: {has_action_values}")
print(f"  - Has node_value computation: {has_node_value}")
print(f"  - Has regret = action_value - node_value: {has_regret_subtraction}")

# Extract the relevant code snippet
lines = source.split('\n')
for i, line in enumerate(lines):
    if 'node_value +=' in line or 'regret =' in line or 'instantaneous' in line.lower():
        print(f"\nLine {i}: {line.strip()}")

passed = has_action_values and has_node_value and has_regret_subtraction
record_test(
    "3. Instantaneous regret formula matches paper",
    passed,
    "r̃(I,a) = v(a) - Σ σ(a')·v(a')" if passed else "Formula not found or incorrect"
)

## Test 4: Linear Weighting in Loss (Algorithm 1, Lines 584-591)

**Paper**: `L(θ) = E[(I,t',r̃) ~ MV][t' · Σ_a (r̃_t'(a) - V(I,a|θ))²]`

**Verification**: Loss function multiplies by iteration t'

In [ ]:
from core.trainer import DeepCFRTrainer

print("=" * 60)
print("TEST 4: Linear Weighting in Loss Function")
print("=" * 60)

# Get source of train_on_samples
source = inspect.getsource(DeepCFRTrainer.train_on_samples)

# Check for linear weighting pattern
has_iter_weights = "iter_weights" in source
has_weighted_loss = "iter_weights * per_sample_loss" in source or \
                    "iter_weights*per_sample_loss" in source
has_linear_weighting_param = "use_linear_weighting" in source

print("\nChecking loss function in train_on_samples:")
print(f"  - Has iteration weights: {has_iter_weights}")
print(f"  - Has weighted loss computation: {has_weighted_loss}")
print(f"  - Has linear_weighting parameter: {has_linear_weighting_param}")

# Find the loss computation lines
lines = source.split('\n')
for i, line in enumerate(lines):
    if 'weighted_loss' in line or 'iter_weights' in line:
        if '=' in line:
            print(f"\nLine {i}: {line.strip()}")

passed = has_iter_weights and has_weighted_loss
record_test(
    "4. Linear weighting (t' * loss) in loss function",
    passed,
    "L = E[t' * Σ(r̃ - V)²]" if passed else "Linear weighting not found"
)

## Test 5: Strategy Network Uses Softmax (Section 5.1)

**Paper**: "In the average strategy network, outputs are interpreted as logits of the probability distribution over actions."

**Verification**: Strategy network output goes through softmax for probabilities

In [ ]:
from core.integration import NetworkMCCFRIntegration

print("=" * 60)
print("TEST 5: Strategy Network Softmax")
print("=" * 60)

# Check if get_strategy_network_probabilities exists and uses softmax
has_strategy_method = hasattr(NetworkMCCFRIntegration, 'get_strategy_network_probabilities')

if has_strategy_method:
    source = inspect.getsource(NetworkMCCFRIntegration.get_strategy_network_probabilities)
    has_softmax = 'softmax' in source.lower() or 'exp' in source
    has_normalize = 'normalize' in source.lower() or '/ sum' in source or '/sum' in source
    
    print(f"\nget_strategy_network_probabilities method found: {has_strategy_method}")
    print(f"Contains softmax or normalization: {has_softmax or has_normalize}")
    
    # Show relevant code
    lines = source.split('\n')
    for i, line in enumerate(lines):
        if 'softmax' in line.lower() or 'normalize' in line.lower() or 'sum' in line:
            print(f"\nLine {i}: {line.strip()}")
    
    passed = has_softmax or has_normalize
else:
    print("WARNING: get_strategy_network_probabilities method not found!")
    passed = False

record_test(
    "5. Strategy network uses softmax for probabilities",
    passed,
    "Logits → softmax → probabilities" if passed else "Softmax not found"
)

## Test 6: Training Parameters Match Paper (Section 5.2)

In [ ]:
from core.deep_cfr import DeepCFR
from core.trainer import DeepCFRTrainer

print("=" * 60)
print("TEST 6: Training Parameters")
print("=" * 60)

# Get default parameters from DeepCFR signature
sig = inspect.signature(DeepCFR.__init__)
defaults = {k: v.default for k, v in sig.parameters.items() if v.default is not inspect.Parameter.empty}

print("\nDeepCFR defaults vs Paper requirements:")
print("="*50)

paper_params = {
    'learning_rate': (0.001, 0.001, "Learning rate"),
    'batch_size': (10000, 10000, "Batch size (paper: 10K-20K)"),
    'sgd_iterations': (4000, 4000, "SGD iterations (paper: 4K-32K)"),
    'train_every': (1, 1, "Train every N iterations (paper: every iter)"),
    'use_network_after': (0, 0, "Use network after N (paper: from iter 1)"),
}

all_match = True
for param, (expected, paper_val, desc) in paper_params.items():
    actual = defaults.get(param, 'NOT FOUND')
    match = actual == expected
    all_match = all_match and match
    status = "OK" if match else "MISMATCH"
    print(f"  {param}: {actual} (expected: {expected}) [{status}]")
    print(f"    -> {desc}")

record_test(
    "6. Training parameters match paper",
    all_match,
    "All parameters match" if all_match else "Some parameters differ"
)

## Test 7: Network Reinitialization (Section 5.2)

**Paper**: "The value model is trained from scratch each CFR iteration, starting from a random initialization."

In [ ]:
print("=" * 60)
print("TEST 7: Network Reinitialization")
print("=" * 60)

# Check that reinitialize_network exists and is called
has_reinit = hasattr(DeepCFR, 'reinitialize_network')
print(f"\nreinitialize_network method exists: {has_reinit}")

if has_reinit:
    # Check _train_network calls reinitialize
    source = inspect.getsource(DeepCFR._train_network)
    calls_reinit = 'reinitialize' in source
    print(f"_train_network calls reinitialize: {calls_reinit}")
    
    # Test actual reinitialization
    deep_cfr = DeepCFR(network_dim=64, traversals_per_iter=10)
    
    # Get initial weights
    initial_weights = deep_cfr.networks[0].action_head.weight.clone()
    
    # Reinitialize
    deep_cfr.reinitialize_network(0)
    
    # Get new weights
    new_weights = deep_cfr.networks[0].action_head.weight
    
    # Check weights changed
    weights_changed = not torch.allclose(initial_weights, new_weights)
    print(f"Weights changed after reinit: {weights_changed}")
    
    passed = has_reinit and calls_reinit and weights_changed
else:
    passed = False

record_test(
    "7. Network reinitialized from scratch each iteration",
    passed,
    "Networks reinit with fresh random weights" if passed else "Reinitialization issue"
)

## Test 8: Gradient Clipping (Section 5.2)

**Paper**: "gradient norm clipping to 1"

In [ ]:
print("=" * 60)
print("TEST 8: Gradient Clipping")
print("=" * 60)

# Check trainer source for gradient clipping
source = inspect.getsource(DeepCFRTrainer.train_on_samples)

has_clip_grad = 'clip_grad_norm' in source
has_max_norm = 'max_norm' in source or 'max_grad_norm' in source

print(f"\nGradient clipping in train_on_samples:")
print(f"  - Uses clip_grad_norm: {has_clip_grad}")
print(f"  - Has max_norm parameter: {has_max_norm}")

# Check default max_grad_norm
trainer_sig = inspect.signature(DeepCFRTrainer.__init__)
trainer_defaults = {k: v.default for k, v in trainer_sig.parameters.items() if v.default is not inspect.Parameter.empty}
max_grad_norm = trainer_defaults.get('max_grad_norm', 'NOT FOUND')
print(f"  - Default max_grad_norm: {max_grad_norm} (paper: 1.0)")

passed = has_clip_grad and max_grad_norm == 1.0
record_test(
    "8. Gradient clipping to norm 1",
    passed,
    f"clip_grad_norm with max_norm={max_grad_norm}" if passed else "Gradient clipping issue"
)

## Test 9: Reservoir Sampling (Algorithm 1, Line 579)

**Paper**: "Initialize reservoir-sampled advantage memories MV,1, MV,2 and strategy memory MΠ."

In [ ]:
print("=" * 60)
print("TEST 9: Reservoir Sampling")
print("=" * 60)

# Check trainer has reservoir sampling
source = inspect.getsource(DeepCFRTrainer.add_sample)

has_reservoir = 'reservoir' in source.lower() or 'total_samples_seen' in source
has_replace_prob = 'replace_prob' in source or 'memory_limit' in source

print(f"\nReservoir sampling in add_sample:")
print(f"  - Has reservoir sampling logic: {has_reservoir}")
print(f"  - Has replacement probability: {has_replace_prob}")

# Show the logic
lines = source.split('\n')
for i, line in enumerate(lines):
    if 'memory_limit' in line or 'replace' in line or 'total_samples' in line:
        print(f"\nLine {i}: {line.strip()}")

passed = has_reservoir or has_replace_prob
record_test(
    "9. Reservoir sampling for memory management",
    passed,
    "Uses reservoir sampling when memory full" if passed else "Reservoir sampling not found"
)

## Test 10: End-to-End Sanity Test

Run a few iterations and verify the system works as expected.

In [ ]:
print("=" * 60)
print("TEST 10: End-to-End Sanity Test (3 iterations)")
print("=" * 60)

# Create DeepCFR with small parameters for testing
deep_cfr = DeepCFR(
    network_dim=64,
    traversals_per_iter=50,  # Small for testing
    sgd_iterations=100,      # Small for testing
    batch_size=32,           # Small for testing
    train_every=1,           # Train every iteration
    use_network_after=0      # Use network from start
)

print("\nRunning 3 iterations...")
results = []

for i in range(3):
    result = deep_cfr.run_iteration()
    results.append(result)
    print(f"\nIteration {i+1}:")
    print(f"  - Utility: {result.get('utility', 'N/A'):.4f}")
    print(f"  - P0 samples: {result.get('new_samples_p0', 0)}")
    print(f"  - P1 samples: {result.get('new_samples_p1', 0)}")
    print(f"  - Used network: {result.get('used_network', False)}")
    if 'loss_p0' in result:
        print(f"  - Loss P0: {result['loss_p0']:.2f}")
    if 'loss_p1' in result:
        print(f"  - Loss P1: {result['loss_p1']:.2f}")

In [ ]:
# Verify key properties
print("\nVerifying end-to-end properties:")

# 1. Network was used
network_used = all(r.get('used_network', False) for r in results)
print(f"  - Network used in all iterations: {network_used}")

# 2. Samples collected for both players
p0_samples = sum(r.get('new_samples_p0', 0) for r in results)
p1_samples = sum(r.get('new_samples_p1', 0) for r in results)
samples_collected = p0_samples > 0 and p1_samples > 0
print(f"  - Samples collected: P0={p0_samples}, P1={p1_samples}")

# 3. Training happened (check if loss exists)
training_happened = any('loss_p0' in r for r in results)
print(f"  - Training happened: {training_happened}")

# 4. Losses are finite (not NaN or Inf)
losses = [r.get('loss_p0', 0) for r in results if 'loss_p0' in r]
losses_finite = all(np.isfinite(l) for l in losses) if losses else True
print(f"  - Losses are finite: {losses_finite}")

passed = network_used and samples_collected and training_happened and losses_finite
record_test(
    "10. End-to-end training runs correctly",
    passed,
    f"Network used, {p0_samples+p1_samples} samples, training OK" if passed else "End-to-end issues"
)

## Summary: Paper Compliance Checklist

In [ ]:
print("\n" + "=" * 60)
print("PAPER COMPLIANCE SUMMARY")
print("=" * 60)

passed_count = sum(1 for r in TEST_RESULTS.values() if r['passed'])
total_count = len(TEST_RESULTS)

print(f"\nResults: {passed_count}/{total_count} tests passed\n")

for name, result in TEST_RESULTS.items():
    status = "PASS" if result['passed'] else "FAIL"
    print(f"[{status}] {name}")
    if result['details']:
        print(f"        {result['details']}")

print("\n" + "=" * 60)
if passed_count == total_count:
    print("ALL TESTS PASSED - Implementation matches paper!")
else:
    print(f"WARNING: {total_count - passed_count} tests failed - review implementation")
print("=" * 60)